# SQL Analytics Toolkit — Demo Notebook

Este notebook muestra todas las capacidades del toolkit con visualizaciones.

**Instalación previa:**
```bash
pip install duckdb pandas matplotlib seaborn plotly
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from toolkit import SQLAnalyticsToolkit

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 110

tk = SQLAnalyticsToolkit()
tk.seed_demo_data(n_users=500, n_orders=3000)

## 1. Executive Summary — KPIs

In [ ]:
summary = tk.executive_summary()

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
kpis = [
    ('Total Revenue', f"${summary['total_revenue']:,.0f}", '#2563EB'),
    ('Avg Order Value', f"${summary['avg_order_value']:,.2f}", '#16A34A'),
    ('Return Rate', f"{summary['return_rate_pct']}%", '#DC2626'),
]
for ax, (label, value, color) in zip(axes, kpis):
    ax.text(0.5, 0.6, value, ha='center', va='center', fontsize=28,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=12,
            color='gray', transform=ax.transAxes)
    ax.axis('off')
    ax.set_facecolor('#F8FAFC')

plt.suptitle('Executive KPIs — 2023', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(summary)

## 2. Cohort Retention Heatmap

In [ ]:
cohort = tk.cohort_retention()

pivot = cohort.pivot(
    index='cohort_month',
    columns='month_number',
    values='retention_pct'
)
pivot.index = pivot.index.astype(str).str[:7]  # YYYY-MM
pivot.columns = [f'Mes {c}' for c in pivot.columns]

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='YlGnBu',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '% retención'},
    ax=ax
)
ax.set_title('Retención mensual por cohorte (%)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Meses desde registro')
ax.set_ylabel('Cohorte (mes de registro)')
plt.tight_layout()
plt.show()

## 3. RFM Segmentation

In [ ]:
rfm = tk.rfm_segmentation()

# Distribución de segmentos
seg_counts = rfm['segment'].value_counts().reset_index()
seg_counts.columns = ['segment', 'count']

colors = {
    'Champions': '#16A34A',
    'Loyal Customers': '#2563EB',
    'Potential Loyalists': '#7C3AED',
    'New Customers': '#0891B2',
    'At Risk': '#D97706',
    'Lost': '#DC2626',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie
wedge_colors = [colors.get(s, '#999') for s in seg_counts['segment']]
ax1.pie(
    seg_counts['count'], labels=seg_counts['segment'],
    colors=wedge_colors, autopct='%1.1f%%',
    startangle=140, pctdistance=0.8
)
ax1.set_title('Distribución de segmentos RFM', fontweight='bold')

# Scatter: Recency vs Monetary
for seg, grp in rfm.groupby('segment'):
    ax2.scatter(
        grp['recency'], grp['monetary'],
        label=seg, alpha=0.6, s=40,
        color=colors.get(seg, '#999')
    )
ax2.set_xlabel('Recency (días desde última compra)')
ax2.set_ylabel('Monetary ($ total gastado)')
ax2.set_title('RFM — Recency vs Monetary', fontweight='bold')
ax2.legend(fontsize=8, bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

## 4. Conversion Funnel

In [ ]:
funnel = tk.conversion_funnel()

fig = go.Figure(go.Funnel(
    y=funnel['event_type'],
    x=funnel['users'],
    textposition='inside',
    textinfo='value+percent initial',
    marker={
        'color': ['#2563EB','#3B82F6','#60A5FA','#93C5FD','#BFDBFE']
    }
))
fig.update_layout(
    title={'text': 'Funnel de Conversión E-commerce', 'x': 0.5, 'font': {'size': 16}},
    height=400,
    margin=dict(l=120, r=40, t=60, b=20)
)
fig.show()
print(funnel[['event_type','users','pct_of_top','pct_prev_step']].to_string(index=False))

## 5. Revenue Breakdown por Categoría

In [ ]:
rev = tk.revenue_breakdown()
rev['month'] = rev['month'].astype(str).str[:7]

pivot_rev = rev.pivot(index='month', columns='category', values='revenue').fillna(0)

fig, ax = plt.subplots(figsize=(13, 5))
pivot_rev.plot(
    kind='bar', stacked=True, ax=ax,
    colormap='tab10', width=0.75
)
ax.set_title('Revenue mensual por categoría', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Categoría', bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout()
plt.show()

## 6. Window Functions — Top clientes por país

In [ ]:
wf = tk.window_functions_demo()

# Top 3 por país
top3 = wf[wf['country_rank'] <= 3].copy()

fig = px.bar(
    top3,
    x='country', y='total_spent',
    color='plan',
    barmode='group',
    hover_data=['user_id','total_orders','percentile'],
    labels={'total_spent': 'Total gastado ($)', 'country': 'País'},
    title='Top 3 clientes por país (total gastado)',
    color_discrete_map={'free':'#94A3B8','basic':'#3B82F6','pro':'#16A34A'}
)
fig.update_layout(height=420)
fig.show()

print('\n── Distribución por cuartil de gasto ──')
print(wf.groupby('quartile')['total_spent'].agg(['min','mean','max','count']).round(2))

## 7. Simulador de Impacto en Conversión

Modificá los valores de `improved_*` para ver cuánto revenue adicional generaría mejorar cada etapa del funnel.

In [ ]:
from toolkit import simulate_funnel_impact

# ── Tasas actuales (del funnel real) ──
# add_to_cart:     474/500 = 0.948
# checkout_start:  349/474 = 0.736
# payment_info:    201/349 = 0.576  <- mayor problema
# purchase:         96/201 = 0.478

# Escenario: simplificamos formulario y agregamos Mercado Pago
df_sim, kpis = simulate_funnel_impact(
    visitors=500,
    avg_order_value=397.48,
    improved_checkout_rate=0.82,
    improved_payment_rate=0.72,
)
print(df_sim.to_string(index=False))

In [ ]:
import numpy as np

stages = df_sim['etapa'].tolist()
x = np.arange(len(stages))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.bar(x - width/2, df_sim['usuarios_actual'],   width, label='Actual',   color='#94A3B8')
bars2 = ax1.bar(x + width/2, df_sim['usuarios_mejorado'], width, label='Mejorado', color='#2563EB')
ax1.set_xticks(x)
ax1.set_xticklabels(stages, rotation=20, ha='right')
ax1.set_ylabel('Usuarios')
ax1.set_title('Usuarios por etapa: actual vs mejorado', fontweight='bold')
ax1.legend()
for bar in bars2:
    ax1.annotate(f'{int(bar.get_height())}',
                 xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 4), textcoords='offset points',
                 ha='center', fontsize=9, color='#2563EB')

labels = ['Revenue actual', 'Revenue mejorado']
values = [kpis['revenue_actual'], kpis['revenue_mejorado']]
colors_bar = ['#94A3B8', '#16A34A']
bars = ax2.bar(labels, values, color=colors_bar, width=0.4)
ax2.set_ylabel('Revenue ($)')
ax2.set_title(f"Impacto en revenue (+{kpis['uplift_pct']}%)", fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'${val:,.0f}', ha='center', fontsize=11, fontweight='bold')
ax2.annotate(
    f"+${kpis['revenue_extra']:,.0f} extra",
    xy=(1, kpis['revenue_mejorado']),
    xytext=(0.5, kpis['revenue_mejorado'] * 0.6),
    fontsize=12, color='#16A34A', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#16A34A')
)
plt.suptitle('Simulador de impacto — Funnel de conversion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparativa de 3 escenarios
escenarios = {
    'Sin cambios':        dict(),
    'Solo formulario':    dict(improved_payment_rate=0.70),
    'Formulario + Meli':  dict(improved_checkout_rate=0.82, improved_payment_rate=0.72),
    'Optimizacion total': dict(improved_cart_rate=0.97, improved_checkout_rate=0.85,
                               improved_payment_rate=0.75, improved_purchase_rate=0.60),
}

resultados = []
for nombre, params in escenarios.items():
    _, kpi = simulate_funnel_impact(visitors=500, avg_order_value=397.48, **params)
    resultados.append({
        'Escenario': nombre,
        'Compras': kpi['compras_mejorado'],
        'Revenue ($)': kpi['revenue_mejorado'],
        'Revenue extra ($)': kpi['revenue_extra'],
        'Uplift (%)': kpi['uplift_pct'],
    })

df_esc = pd.DataFrame(resultados)

fig, ax = plt.subplots(figsize=(11, 4))
colors_esc = ['#94A3B8', '#60A5FA', '#2563EB', '#16A34A']
bars = ax.barh(df_esc['Escenario'], df_esc['Revenue ($)'], color=colors_esc)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.set_title('Revenue por escenario de mejora', fontsize=13, fontweight='bold')
for bar, row in zip(bars, df_esc.itertuples()):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f"  {row.Compras} compras  |  +{row._5}%",
            va='center', fontsize=10)
ax.set_xlim(0, df_esc['Revenue ($)'].max() * 1.3)
plt.tight_layout()
plt.show()
print(df_esc.to_string(index=False))